In [1]:
import numpy as np
import pandas as pd

# ====== 1) 读取数据 ======
path_T = r"E:\00_Commute_Scenario_Research\data\03_processed\T_observed.csv"
path_O = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-O端回写后.csv"
path_D = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-D端回写后.csv"

col_O = "O人数行业:IT通信电子"
col_D = "D人数行业:IT通信电子"

T = pd.read_csv(path_T)
# T_observed 是宽矩阵，列名即目的地编号
dest_ids = pd.Index([int(c) for c in T.columns], name="taz")
# 行编号按列编号顺序对齐（你的 T_observed 通常是方阵）
if T.shape[0] != len(dest_ids):
    raise ValueError(f"T_observed 不是方阵，shape={T.shape}")
orig_ids = dest_ids.copy()

support = (T.to_numpy() > 0)   # 允许边 mask, shape n x n
n = support.shape[0]

Odf = pd.read_csv(path_O)
Ddf = pd.read_csv(path_D)

Ovec = (
    Odf.assign(taz=Odf["taz"].astype(int))
       .set_index("taz")[col_O]
       .reindex(orig_ids, fill_value=0.0)
       .astype(float)
       .to_numpy()
)
Dvec = (
    Ddf.assign(taz=Ddf["taz"].astype(int))
       .set_index("taz")[col_D]
       .reindex(dest_ids, fill_value=0.0)
       .astype(float)
       .to_numpy()
)

# 全局配平（若总量不等，先缩放一侧）
sumO, sumD = Ovec.sum(), Dvec.sum()
if sumO > 0 and sumD > 0 and abs(sumO - sumD) > 1e-8:
    Dvec = Dvec * (sumO / sumD)

# ====== 2) 你提出的“平均分配法”失衡检查 ======
k = support.sum(axis=1)  # 每个起点可达终点数
avg = np.divide(Ovec, k, out=np.zeros_like(Ovec), where=(k > 0))

# implied_D[j] = 所有能到 j 的起点平均分配累加
implied_D = (support * avg[:, None]).sum(axis=0)
diff_D = implied_D - Dvec

check_df = pd.DataFrame({
    "taz": dest_ids,
    "D_target": Dvec,
    "D_implied_by_equal_split": implied_D,
    "diff": diff_D,
    "abs_diff": np.abs(diff_D),
})
print("平均分配法 - 终点端最大绝对误差:", check_df["abs_diff"].max())
print("平均分配法 - 终点端MAE:", check_df["abs_diff"].mean())

# 组合方式（按起点可达终点集合）统计
sig_series = pd.Series(
    [tuple(np.where(support[i])[0].tolist()) for i in range(n)],
    index=orig_ids,
    name="signature"
)
combo = pd.DataFrame({"signature": sig_series, "O_i": Ovec})
combo_stat = combo.groupby("signature", dropna=False).agg(
    n_origins=("O_i", "size"),
    O_total=("O_i", "sum")
).reset_index()

# 给每个signature算“该signature覆盖终点的D总量”（注意会有重叠，不可直接当守恒）
def d_covered(sig):
    if len(sig) == 0:
        return 0.0
    return Dvec[np.array(sig, dtype=int)].sum()

combo_stat["D_covered"] = combo_stat["signature"].apply(d_covered)
combo_stat["O_minus_Dcovered"] = combo_stat["O_total"] - combo_stat["D_covered"]

# ====== 3) 正确分配: IPF/RAS（在 support 上） ======
# 初值可用 T_observed 值（更贴近现实），并给允许边一个极小正数避免数值问题
X = np.where(support, T.to_numpy().astype(float) + 1e-12, 0.0)

# 若某行/列在 support 上完全无边，但目标值>0，则无解
bad_rows = np.where((support.sum(axis=1) == 0) & (Ovec > 1e-12))[0]
bad_cols = np.where((support.sum(axis=0) == 0) & (Dvec > 1e-12))[0]
if len(bad_rows) or len(bad_cols):
    raise ValueError(f"存在无连接但目标>0的点: rows={len(bad_rows)}, cols={len(bad_cols)}")

max_iter = 5000
tol = 1e-8

for it in range(max_iter):
    # 行缩放
    row_sum = X.sum(axis=1)
    r = np.divide(Ovec, row_sum, out=np.ones_like(Ovec), where=(row_sum > 0))
    X = X * r[:, None]
    X[~support] = 0.0

    # 列缩放
    col_sum = X.sum(axis=0)
    c = np.divide(Dvec, col_sum, out=np.ones_like(Dvec), where=(col_sum > 0))
    X = X * c[None, :]
    X[~support] = 0.0

    # 收敛检查
    err_row = np.max(np.abs(X.sum(axis=1) - Ovec))
    err_col = np.max(np.abs(X.sum(axis=0) - Dvec))
    if max(err_row, err_col) < tol:
        print(f"IPF收敛: iter={it}, row_err={err_row:.3e}, col_err={err_col:.3e}")
        break
else:
    print("IPF未在迭代上限内收敛")

# 结果检查
print("IPF后 行约束最大误差:", np.max(np.abs(X.sum(axis=1) - Ovec)))
print("IPF后 列约束最大误差:", np.max(np.abs(X.sum(axis=0) - Dvec)))

# 导出长表
ii, jj = np.where(support)
flow_long = pd.DataFrame({
    "o_taz": orig_ids.to_numpy()[ii],
    "d_taz": dest_ids.to_numpy()[jj],
    "flow_it": X[ii, jj]
})
# flow_long.to_csv("IT_flow_on_observed_support.csv", index=False, encoding="utf-8-sig")

平均分配法 - 终点端最大绝对误差: 1776.0391376131931
平均分配法 - 终点端MAE: 38.865352476075785
IPF收敛: iter=51, row_err=8.212e-09, col_err=4.547e-13
IPF后 行约束最大误差: 8.211827662307769e-09
IPF后 列约束最大误差: 4.547473508864641e-13


In [2]:
# 导入数值计算库
import numpy as np
# 导入表格处理库
import pandas as pd

# ====== 1) 读取数据 ======

# 指定原生观测通勤矩阵路径（宽矩阵：行是O，列是D）
path_T = r"E:\00_Commute_Scenario_Research\data\03_processed\T_observed.csv"
# 指定拉比例后O端结果路径
path_O = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-O端回写后.csv"
# 指定拉比例后D端结果路径
path_D = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-D端回写后.csv"

# 指定要分析的行业列（O端）
col_O = "O人数行业:IT通信电子"
# 指定要分析的行业列（D端）
col_D = "D人数行业:IT通信电子"

# 读取原生观测矩阵
T = pd.read_csv(path_T)

# 将列名转成整数TAZ编号，作为D端ID
dest_ids = pd.Index([int(c) for c in T.columns], name="taz")

# 检查是否方阵（行数是否等于列数）
if T.shape[0] != len(dest_ids):
    # 如果不是方阵则报错停止
    raise ValueError(f"T_observed 不是方阵，shape={T.shape}")

# 假设行TAZ编号与列TAZ编号一一对应（当前项目约定）
orig_ids = dest_ids.copy()

# 构造支持集矩阵（True表示该O-D在原生矩阵中非零，允许连边）
support = (T.to_numpy() > 0)

# 记录TAZ规模
n = support.shape[0]

# 读取拉比例后O端表
Odf = pd.read_csv(path_O)
# 读取拉比例后D端表
Ddf = pd.read_csv(path_D)

# 组装O端IT行业向量，并按orig_ids重排对齐
Ovec = (
    Odf.assign(taz=Odf["taz"].astype(int))
       .set_index("taz")[col_O]
       .reindex(orig_ids, fill_value=0.0)
       .astype(float)
       .to_numpy()
)

# 组装D端IT行业向量，并按dest_ids重排对齐
Dvec = (
    Ddf.assign(taz=Ddf["taz"].astype(int))
       .set_index("taz")[col_D]
       .reindex(dest_ids, fill_value=0.0)
       .astype(float)
       .to_numpy()
)

# 计算O端总量
sumO = Ovec.sum()
# 计算D端总量
sumD = Dvec.sum()

# 如果两端总量不一致，则将D端按比例缩放到与O端一致（全局配平）
if sumO > 0 and sumD > 0 and abs(sumO - sumD) > 1e-8:
    Dvec = Dvec * (sumO / sumD)

# ====== 2) 原生OD组合方式详细分析 ======

# 计算每个O有多少个非零D连接（组合数）
d_count_per_o = support.sum(axis=1)

# 构造每个O对应的D列表（使用真实TAZ编号，不是列位置）
d_list_per_o = [dest_ids[np.where(support[i])[0]].tolist() for i in range(n)]

# 构造每个O的组合明细表
origin_combo_df = pd.DataFrame({
    "o_taz": orig_ids.to_numpy(),
    "d_count": d_count_per_o,
    "d_list": d_list_per_o,
    "O_it": Ovec
})

# 为每个O统计其在原生矩阵上的总流量（所有D求和）
origin_combo_df["T_row_sum"] = T.to_numpy().sum(axis=1)

# 输出基础摘要
print("\n========== 原生OD组合方式基础统计 ==========")
print("TAZ总数:", n)
print("有出边的O数量:", int((d_count_per_o > 0).sum()))
print("无出边的O数量:", int((d_count_per_o == 0).sum()))
print("每个O连接D数量描述:")
print(origin_combo_df["d_count"].describe())

# 输出连接数最高的前20个O
print("\n连接D数量最多的前20个O:")
print(origin_combo_df.sort_values("d_count", ascending=False).head(20)[["o_taz", "d_count", "O_it", "T_row_sum"]])

# ====== 3) 你原来提出的“平均分配法”误差检查 ======

# 平均分配分母：每个O的可达D数量
k = d_count_per_o

# 平均分配值：O_i / k_i（仅k_i>0时）
avg = np.divide(Ovec, k, out=np.zeros_like(Ovec), where=(k > 0))

# 按平均分配推导得到D端值（把所有能到j的O贡献相加）
implied_D = (support * avg[:, None]).sum(axis=0)

# 与目标D端差值
diff_D = implied_D - Dvec

# 构造平均分配误差表
check_df = pd.DataFrame({
    "d_taz": dest_ids.to_numpy(),
    "D_target": Dvec,
    "D_implied_by_equal_split": implied_D,
    "diff": diff_D,
    "abs_diff": np.abs(diff_D),
})

# 输出误差统计
print("\n========== 平均分配法误差 ==========")
print("终点端最大绝对误差:", check_df["abs_diff"].max())
print("终点端MAE:", check_df["abs_diff"].mean())

# 输出误差最大的前20个D
print("\n平均分配法误差最大的前20个D:")
print(check_df.sort_values("abs_diff", ascending=False).head(20))

# ====== 4) 组合方式签名分析（按同一D集合归类O） ======

# 构造签名（每个O对应一个D集合元组，元素是D端真实TAZ编号）
sig_series = pd.Series(
    [tuple(dest_ids[np.where(support[i])[0]].tolist()) for i in range(n)],
    index=orig_ids,
    name="signature"
)

# 组合表：每个O属于哪个签名
combo = pd.DataFrame({
    "o_taz": orig_ids.to_numpy(),
    "signature": sig_series.values,
    "O_it": Ovec
})

# 定义函数：给定签名，计算该签名下D端IT总量（注意D会在不同签名中重复计入）
def d_covered(sig):
    # 若签名为空，返回0
    if len(sig) == 0:
        return 0.0
    # 将签名中的D taz映射到Dvec并求和
    idx = dest_ids.get_indexer(pd.Index(sig))
    # 过滤未匹配索引
    idx = idx[idx >= 0]
    # 返回D覆盖和
    return Dvec[idx].sum()

# 定义函数：返回签名中D数量
def d_num(sig):
    # 返回签名长度
    return len(sig)

# 定义函数：返回签名文本（防止打印过长，可按需调整截断长度）
def sig_text(sig, max_len=120):
    # 把元组转字符串
    s = str(sig)
    # 超长时截断显示
    return s if len(s) <= max_len else (s[:max_len] + "...")

# 统计每种签名下：O数量、O总量
combo_stat = combo.groupby("signature", dropna=False).agg(
    n_origins=("o_taz", "size"),
    origins=("o_taz", lambda x: list(x)),
    O_it_total=("O_it", "sum")
).reset_index()

# 计算该签名覆盖D端总量
combo_stat["D_it_covered"] = combo_stat["signature"].apply(d_covered)

# 计算差值（你关注的“每种组合方式下IT两端差值”）
combo_stat["IT_diff_O_minus_D"] = combo_stat["O_it_total"] - combo_stat["D_it_covered"]

# 计算签名的D数量
combo_stat["n_dests"] = combo_stat["signature"].apply(d_num)

# 生成可读签名文本
combo_stat["signature_text"] = combo_stat["signature"].apply(sig_text)

# 按绝对差值排序
combo_stat["abs_diff"] = combo_stat["IT_diff_O_minus_D"].abs()
combo_stat = combo_stat.sort_values("abs_diff", ascending=False)

# 输出签名统计摘要
print("\n========== 组合方式签名统计 ==========")
print("签名种类数:", len(combo_stat))
print("差值最大（按绝对值）前20种签名:")
print(combo_stat.head(20)[[
    "n_origins", "n_dests", "O_it_total", "D_it_covered", "IT_diff_O_minus_D", "abs_diff", "signature_text"
]])

# ====== 5) 每个O对应哪些D（你要的明细） ======

# 输出前30个O作为示例（完整表在origin_combo_df里）
print("\n========== 每个O对应哪些D（前30行示例） ==========")
print(origin_combo_df.head(30)[["o_taz", "d_count", "d_list", "O_it"]])

# ====== 6) 保存分析结果（可选） ======

# 输出目录（如不存在可改成本地已有目录）
out_dir = r"E:\00_Commute_Scenario_Research\results\tables"

# 保存每个O的组合明细
origin_combo_df.to_csv(
    out_dir + r"\it_origin_to_dest_combinations.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存平均分配误差表
check_df.to_csv(
    out_dir + r"\it_equal_split_destination_error.csv",
    index=False,
    encoding="utf-8-sig"
)

# 保存签名差值表（你重点要的结果）
combo_stat.to_csv(
    out_dir + r"\it_signature_diff_O_vs_D.csv",
    index=False,
    encoding="utf-8-sig"
)

# 打印保存完成提示
print("\n结果文件已输出到:", out_dir)
print("1) it_origin_to_dest_combinations.csv")
print("2) it_equal_split_destination_error.csv")
print("3) it_signature_diff_O_vs_D.csv")


========== 原生OD组合方式基础统计 ==========
TAZ总数: 2427
有出边的O数量: 2397
无出边的O数量: 30
每个O连接D数量描述:
count    2427.000000
mean      211.501030
std       240.943788
min         0.000000
25%        23.000000
50%       106.000000
75%       344.000000
max      1364.000000
Name: d_count, dtype: float64

连接D数量最多的前20个O:
      o_taz  d_count         O_it  T_row_sum
1010   1010     1364  1166.408040    19170.0
2374   2374     1357  1320.603097    20287.0
1031   1031     1351  1135.773162    17028.0
1475   1475     1342  1076.715329    15655.0
489     489     1261  1064.502935    11617.0
647     647     1205  1063.944415    16640.0
919     919     1200   898.873033     9491.0
475     475     1183  1276.577267    16565.0
916     916     1176   931.899362    12899.0
1752   1752     1150   650.520393     9301.0
1772   1772     1148   563.527040     9080.0
536     536     1143   826.693377    12591.0
530     530     1134   795.389830    11163.0
1198   1198     1116   619.633613     7005.0
532     532     1100   99

In [3]:
import ast
import pandas as pd

path_combo = r"E:\00_Commute_Scenario_Research\results\tables\it_origin_to_dest_combinations.csv"
path_D = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-D端回写后.csv"

combo_df = pd.read_csv(path_combo)
Ddf = pd.read_csv(path_D)

combo_df["d_list"] = combo_df["d_list"].apply(ast.literal_eval)

d_it_map = (
    Ddf.assign(taz=Ddf["taz"].astype(int))
       .set_index("taz")["D人数行业:IT通信电子"]
       .astype(float)
)

combo_df["T_row_sum"] = combo_df["d_list"].apply(
    lambda ds: d_it_map.reindex(ds).fillna(0.0).sum()
)
combo_df["IT_diff_O_minus_Dsum"] = combo_df["O_it"] - combo_df["T_row_sum"]

combo_df.to_csv(path_combo, index=False, encoding="utf-8-sig")

# 泊松

In [4]:
import ast
import numpy as np
import pandas as pd
import statsmodels.api as sm

# 路径
path_T = r"E:\00_Commute_Scenario_Research\data\03_processed\T_observed.csv"
path_O = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-O端回写后.csv"
path_D = r"F:\02_250910_Commute\12-gvc\0409比例拉取结果\general\[主城区]taz4-D端回写后.csv"

# 目标行业
industry_map = {
    "IT通信电子": ("O人数行业:IT通信电子", "D人数行业:IT通信电子"),
    "机械制造": ("O人数行业:机械制造", "D人数行业:机械制造"),
}

# 读取数据
T = pd.read_csv(path_T)
Odf = pd.read_csv(path_O)
Ddf = pd.read_csv(path_D)

# 行列TAZ
taz_ids = pd.Index([int(c) for c in T.columns], name="taz")
support = (T.to_numpy() > 0)
C_support = support.copy()

# 构建距离矩阵时如果你已有 C_matrix，就直接读；否则需要补进来
# 这里假设你已经有同维度距离矩阵 C_matrix.csv
C = pd.read_csv(r"E:\00_Commute_Scenario_Research\data\03_processed\C_matrix.csv").to_numpy()

def build_industry_vectors(industry_name, o_col, d_col):
    O = (
        Odf.assign(taz=Odf["taz"].astype(int))
           .set_index("taz")[o_col]
           .astype(float)
           .reindex(taz_ids, fill_value=0.0)
           .to_numpy()
    )
    D = (
        Ddf.assign(taz=Ddf["taz"].astype(int))
           .set_index("taz")[d_col]
           .astype(float)
           .reindex(taz_ids, fill_value=0.0)
           .to_numpy()
    )

    # 去掉行业人数 < 1 的 taz
    valid = (O >= 1.0) & (D >= 1.0)
    idx = np.where(valid)[0]

    return O, D, idx

def ipf_on_support(O, D, support_mask, base=None, max_iter=5000, tol=1e-8):
    X = np.where(support_mask, 1.0, 0.0) if base is None else np.where(support_mask, base, 0.0)
    X = X.astype(float)

    bad_rows = np.where((support_mask.sum(axis=1) == 0) & (O > 1e-12))[0]
    bad_cols = np.where((support_mask.sum(axis=0) == 0) & (D > 1e-12))[0]
    if len(bad_rows) or len(bad_cols):
        raise ValueError(f"存在无连接但边际>0的点: rows={len(bad_rows)}, cols={len(bad_cols)}")

    # 边际总量不一致时先全局配平
    sumO, sumD = O.sum(), D.sum()
    if sumO > 0 and sumD > 0 and abs(sumO - sumD) > 1e-8:
        D = D * (sumO / sumD)

    for _ in range(max_iter):
        row_sum = X.sum(axis=1)
        r = np.divide(O, row_sum, out=np.ones_like(O), where=row_sum > 0)
        X *= r[:, None]
        X[~support_mask] = 0.0

        col_sum = X.sum(axis=0)
        c = np.divide(D, col_sum, out=np.ones_like(D), where=col_sum > 0)
        X *= c[None, :]
        X[~support_mask] = 0.0

        if max(np.max(np.abs(X.sum(axis=1) - O)), np.max(np.abs(X.sum(axis=0) - D))) < tol:
            break

    return X

results = {}

for ind_name, (o_col, d_col) in industry_map.items():
    O, D, valid_idx = build_industry_vectors(ind_name, o_col, d_col)

    # 取行业有效集合的子矩阵
    sub_support = support[np.ix_(valid_idx, valid_idx)]
    sub_C = C[np.ix_(valid_idx, valid_idx)]
    O_sub = O[valid_idx]
    D_sub = D[valid_idx]
    taz_sub = taz_ids[valid_idx].to_numpy()

    # 用距离作为阻抗构造基底
    # 如果你不想先加距离项，也可以直接用 support_mask
    theta0 = 0.01
    base = np.exp(-theta0 * sub_C)
    base[~sub_support] = 0.0

    # 生成行业 T 矩阵
    X_ind = ipf_on_support(O_sub, D_sub, sub_support, base=base)

    # 展开成长表
    ii, jj = np.where(sub_support)
    flow_df = pd.DataFrame({
        "industry": ind_name,
        "o_taz": taz_sub[ii],
        "d_taz": taz_sub[jj],
        "flow": X_ind[ii, jj],
        "distance": sub_C[ii, jj],
    })
    flow_df["log_offset"] = np.log(np.maximum(O_sub[ii] * D_sub[jj], 1e-12))

    # Poisson 回归
    # 模型：flow ~ distance + offset(log(O_i * D_j))
    X_design = sm.add_constant(flow_df[["distance"]], has_constant="add")
    model = sm.GLM(
        flow_df["flow"],
        X_design,
        family=sm.families.Poisson(),
        offset=flow_df["log_offset"]
    )
    fit = model.fit()

    beta = fit.params["distance"]
    rigidity = -beta

    results[ind_name] = {
        "taz_count": len(valid_idx),
        "flow_df": flow_df,
        "matrix": X_ind,
        "poisson_result": fit,
        "rigidity": rigidity,
        "beta": beta,
    }

    print(f"\n{ind_name}")
    print(f"有效 taz 数: {len(valid_idx)}")
    print(f"Poisson 距离系数 beta: {beta:.6f}")
    print(f"刚性指标 rigidity = -beta: {rigidity:.6f}")
    print(fit.summary())


IT通信电子
有效 taz 数: 1871
Poisson 距离系数 beta: -0.001321
刚性指标 rigidity = -beta: 0.001321
                 Generalized Linear Model Regression Results                  
Dep. Variable:                   flow   No. Observations:               484261
Model:                            GLM   Df Residuals:                   484259
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -2.9549e+05
Date:                Fri, 10 Apr 2026   Deviance:                   5.3608e+05
Time:                        11:40:26   Pearson chi2:                 2.27e+07
No. Iterations:                    79   Pseudo R-squ. (CS):             0.8079
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------